📝
# FieldCare — Sprint 2 Campus notebook

HelioDesk now wants to share its FieldCare service with approved callers. A working endpoint alone does not tell us **who may call it**, **where secrets travel**, or **how much work a caller may trigger**.

In this one notebook, protect your Sprint 1 service, investigate secret boundaries, observe a rate limit, and apply the controls independently to a new caller arrangement.

| Section | Activity | Evidence |
|---|---|---|
| 0. Setup | Import your Sprint 1 source checkpoint | Preserved routes and source |
| 1. Authentication | Add the supplied API-key guard | Allowed/rejected HTTP calls and adapter observations |
| 2. Secret boundaries | Investigate logs, prompts and Git | Safe diagnostic observations using a nonfunctional marker |
| 3. Rate limits | Follow a two-request worked policy | Rejection, isolation and timed renewal |
| 4. Independent practice | Apply three requests per caller | Your predictions and chosen experiment |
| 5. Secured-service checkpoint | Adapt to the weekend team | Independent configuration and evidence |
| 6. Save | Export one final checkpoint | One source/evidence ZIP and your saved notebook |

Use the table of contents to navigate. Run worked examples in order; stop at **Your turn** to implement and explain your choices. An untouched Run all completes demonstrations and reports independent work as `pending`—it does not complete an assessment.

**Before starting:** open this notebook from the course GitHub repository, save a copy in Drive, select a Python CPU runtime and have your Sprint 1 checkpoint ZIP available. Setup downloads the shared code at the configured GitHub release revision and installs it. Upload your personal checkpoint through Colab's Files panel and set its path below. All examples use local demo responses; no provider account, real credential or live model is required.

Use this Campus notebook for every Campus lesson in this sprint. The live sessions use a separate Live notebook. Save one personal copy of each; keep your cumulative project here.


## Campus lessons in Notion

- [API keys in plain language](https://app.notion.com/p/3ea5a55972ad81e898f1ca03d801e980?pvs=204)
- [Adding API key auth to the scaffold](https://app.notion.com/p/3ea5a55972ad81d3951cecd6d8dbd5c7?pvs=204)
- [What is a secret and where it should live](https://app.notion.com/p/3ea5a55972ad81d48e28c56f84342984?pvs=204)
- [Why AI services need rate limits](https://app.notion.com/p/3ea5a55972ad8157a8d4f24c0a8674e2?pvs=204)
- [Applying per-key rate limits](https://app.notion.com/p/3ea5a55972ad81ac920cf6a901d6388c?pvs=204)
- [Secured service checkpoint](https://app.notion.com/p/3ea5a55972ad8177b84fdac0b397fa8d?pvs=204)


## 0. Set up once and import your service

If you used a different source version in this kernel, restart it before setup. Shared helpers handle temporary servers and observation mechanics; the route inventory, guard attachment, caller mapping, policy, request headers and experiment remain visible here.

The setup cell already contains the published course repository and a fixed reviewed revision. Run it unchanged in a fresh Colab runtime. It downloads the shared helpers and preserves existing workspace files on a rerun.

In [ ]:
#@title Load the shared course code from GitHub
from pathlib import Path
import os, re, sys, subprocess, tempfile

# Reviewed shared helpers: this immutable revision keeps every lesson on the same source.
GITHUB_REPO = 'https://github.com/richhiey/ai-app-dev_Mod-B.git'
GITHUB_REF = 'fb9d95b1cadcc29972cb9dd7388201aaa5868e1d'
repo_url = os.environ.get('MODULE_B_REPO_URL', GITHUB_REPO).strip()
repo_ref = os.environ.get('MODULE_B_REPO_REF', GITHUB_REF).strip()
destination = Path(os.environ.get('MODULE_B_REPO', '/content/ai-app-dev_Mod-B')).expanduser()
candidates = [destination, Path.cwd(), *Path.cwd().resolve().parents]
REPO = next((p.resolve() for p in candidates if (p/'src/module_b/notebook.py').is_file()), None)

if REPO is None:
    if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', repo_url):
        raise RuntimeError('The course author must set GITHUB_REPO to the published HTTPS GitHub repository URL.')
    if not re.fullmatch(r'[0-9a-fA-F]{40}|refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*', repo_ref):
        raise RuntimeError('The course author must set GITHUB_REF to a release tag (refs/tags/...) or full commit SHA.')
    if destination.exists():
        raise FileExistsError('The destination already exists without the expected helpers. Preserve it and use a fresh runtime or a new MODULE_B_REPO path.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Prepare separately, so a failed download cannot replace existing student work.
    with tempfile.TemporaryDirectory(prefix='module-b-download-', dir=destination.parent) as temporary:
        checkout = Path(temporary)/'checkout'
        subprocess.run(['git','clone','--no-checkout','--depth','1',repo_url,str(checkout)],check=True)
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin',repo_ref],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
        if not (checkout/'src/module_b/notebook.py').is_file() or not (checkout/'requirements.lock').is_file():
            raise RuntimeError('The selected repository revision does not contain the expected Module B source layout.')
        checkout.rename(destination)
    REPO = destination.resolve()

# Existing local checkouts are reused; setup never pulls, resets or overwrites them.
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'-c',str(REPO/'requirements.lock')],check=True)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
from module_b import __version__
print('Shared package ready:', __version__)
print('Source checkout:', REPO)

In [ ]:
import json
import httpx
from module_b.data import load_fixture
from module_b.edits import write_source
from module_b.security import load_callers
from module_b.security_lab import (prepare_security_workspace, runtime_keys, demo_runtime,
    observe_security, review_security, configuration_template_ok)
from module_b.secret_lab import MARKER, marker_present, repository_rehearsal
from module_b.notebook import (table, baseline, register_source, apply_student_edits,
    revision, fresh, export_progress)
from module_b.workspace import restore_workspace

### One notebook, named stages that preserve your work

`auth` holds the worked authentication extension. `limit_lab` is its worked two-request example. `practice` is your independent implementation; `assessment` is created only when that implementation passes the local checks. Copies retain the same incoming service, so earlier demonstration cells never reset a later policy.

Set `SPRINT1_CHECKPOINT` to your saved Sprint 1 ZIP (or local project directory). If it is unavailable, `USE_DEMO_RECOVERY=True` explicitly selects the supplied `/v1/diagnose` and `/v2/diagnose` example. Recovery helps you learn the controls; it does not claim your Sprint 1 assessment was completed.

To resume a Sprint 2 attempt in a fresh runtime, also set `RESUME_ZIP` and `RESUME_STAGE` to the stage printed at your last export. Keep the Sprint 1 input available to rebuild the earlier worked stages. Reruns keep existing files. Choose a new `RUN_NAME` only for a deliberately separate attempt.

In [ ]:
RUN_NAME = 'sprint-2'
SPRINT1_CHECKPOINT = ''
USE_DEMO_RECOVERY = True
RESUME_ZIP = ''
RESUME_STAGE = 'practice'  # auth, practice or assessment, as printed at export
work = REPO/'work'/RUN_NAME
if RESUME_ZIP:
    assert RESUME_STAGE in ('auth','practice','assessment')
    if not (work/RESUME_STAGE).exists():
        restore_workspace(RESUME_ZIP,work/RESUME_STAGE)
if not SPRINT1_CHECKPOINT and not USE_DEMO_RECOVERY:
    raise ValueError('Set your Sprint 1 checkpoint path, or explicitly choose demo recovery.')
auth, auth_origin = prepare_security_workspace(work/'auth',checkpoint=SPRINT1_CHECKPOINT or None,repo_root=REPO)
accepted = load_fixture(auth,'valid-request')
CALLER_ENV = {'dispatch':'FIELDCARE_DISPATCH_KEY','partner':'FIELDCARE_PARTNER_KEY'}
evidence = {}
print('Starting source:',auth_origin['provenance'])
print('Authentication stage:',auth)

## 1. Worked example — admit recognized callers

**Authentication** checks a presented credential and maps it to a configured caller label. Possession of a shared key does not identify the individual person using it. An **API key** is a secret presented with a request. This teaching policy lets either recognized caller use every registered diagnostic POST operation. It does not provide roles, per-equipment permissions, user login or a complete identity system.

Keep two boundaries separate: the caller sends `X-API-Key` to FieldCare; the provider credential would belong only to FieldCare's model adapter. The caller key must not enter the question, prompt, logs or response.

Write environment-variable **names** into source. Actual temporary values will be supplied only to the child server at runtime. `POLICY=None` means authentication without a usage limit for this first stage.

In [ ]:
auth_settings = "CALLER_ENV = {'dispatch':'FIELDCARE_DISPATCH_KEY','partner':'FIELDCARE_PARTNER_KEY'}\nPOLICY = None\n"
write_source(auth,'app/security_settings.py',auth_settings)
print(auth_settings)

### Attach after all route registrations

A **middleware** guard runs around incoming requests. Inspect the final `app/main.py` source after the following cell. The protected inventory must be collected **after** every `include_router` call, including the route you added in Sprint 1. If you add another route later, move its registration above this guard and recheck coverage.

Expected behavior: recognized keys proceed; missing or wrong keys receive `401` before application/model work. Public GET health and documentation remain callable. The supplied helper requires complete, distinct, non-placeholder runtime values and refuses startup when configuration is incomplete.

In [ ]:
attachment = '\nfrom module_b.security import SecurityMiddleware, protected_post_paths\nfrom app.security_settings import CALLER_ENV, POLICY\napp.add_middleware(SecurityMiddleware, caller_env=CALLER_ENV,\n                   protected_paths=protected_post_paths(app), policy=POLICY)\n'
register_source(auth,attachment)
print((auth/'app/main.py').read_text())

In [ ]:
env = runtime_keys(CALLER_ENV)
print('Configured labels:',sorted(load_callers(CALLER_ENV,env)))  # Never print env or request headers.

### Predict, then send actual HTTP requests

For each inherited path, predict the status for dispatch, partner, missing and wrong credentials. The visible request shows where the header belongs. We retain only labels, paths and statuses; never credential values.

The owned server stops at the end of the `with` block. These are real local HTTP requests with demo answers, not a deployed or live-provider service.

In [ ]:
inventory = observe_security(auth,caller_env=CALLER_ENV,sequence=[])
assert inventory.get('startup')=='ready', inventory
paths = inventory['routes']
auth_http = []
with demo_runtime(auth,env) as service:
    api = httpx.get(service.base_url+'/openapi.json').json()
    documented_paths = sorted(p for p,methods in api['paths'].items() if 'post' in methods)
    assert set(documented_paths)<=set(paths)  # Hidden-from-docs POST routes remain in the full inventory.
    for path in paths:
        for caller in (*CALLER_ENV,'missing','wrong'):
            headers = {} if caller=='missing' else {'X-API-Key':env[CALLER_ENV[caller]] if caller in CALLER_ENV else 'invalid-local-test'}
            response = httpx.post(service.base_url+path,json=accepted,headers=headers)
            auth_http.append({'path':path,'caller':caller,'status':response.status_code})
    public = {p:httpx.get(service.base_url+p).status_code for p in ('/health','/docs','/openapi.json')}
table(auth_http)
print('Public GET:',public)
assert all(r['status']==(200 if r['caller'] in CALLER_ENV else 401) for r in auth_http)
assert all(status==200 for status in public.values())

### Check the work boundary separately

Status alone does not prove rejected requests avoided model work. A **separate in-process probe** counts adapter entries on new requests and blocks external network calls. It is not attached to the HTTP requests above.

Expect one demo adapter entry for each supported authorized request and zero for rejected callers. Then remove each configured runtime variable in a separate worker; expect startup refusal. This fails closed instead of silently admitting callers when a key is absent.

The cell prints a safe refusal record for each missing variable, then starts an actual HTTP service with the complete environment again. Expect the supported dispatch request to succeed after restoration.

In [ ]:
auth_sequence = [{'caller':caller,'path':path} for path in paths for caller in (*CALLER_ENV,'missing','wrong')]
auth_observed = observe_security(auth,caller_env=CALLER_ENV,sequence=auth_sequence)
table(auth_observed['rows'],['caller','path','status','adapter_calls'])
assert auth_observed['routes']==paths
assert all(row['adapter_calls']==int(row['caller'] in CALLER_ENV) for row in auth_observed['rows'])
missing_configuration = []
for name in CALLER_ENV.values():
    refused = observe_security(auth,caller_env=CALLER_ENV,sequence=[],missing_env=name)
    missing_configuration.append({'missing_name':name,'startup':refused.get('startup')})
table(missing_configuration)
assert all(row['startup']=='refused_missing_or_invalid_caller_configuration' for row in missing_configuration)
with demo_runtime(auth,env) as service:
    restored = httpx.post(service.base_url+'/v1/diagnose',json=accepted,headers={'X-API-Key':env[CALLER_ENV['dispatch']]})
    assert restored.status_code==200
print('Complete runtime configuration restored:',restored.status_code)
evidence['authentication'] = {'http':auth_http,'probe':auth_observed,'missing_configuration':missing_configuration,'recovered_http_status':restored.status_code,'source_revision':revision(auth)}

### Apply the mental model

A colleague says, “The request returned `401`, so it must have consumed one of the caller's model calls.” Identify what is missing from that claim and use the observations above to correct it.

<details><summary>Solution</summary>

**Result you should get:** the unrecognized request did not reach the adapter. **How:** match the caller/path to the separate probe's zero adapter entries. There is no recognized caller bucket to charge; the later usage policy applies after authentication.

</details>

## 2. Worked investigations — keep secrets inside their boundary

A credential can leak without appearing in the response. Logging a whole request, copying request metadata into a prompt or committing a configuration file can carry it outside its intended boundary.

The following experiments use a **nonfunctional teaching marker**, never a real key. Predict whether each representation contains the marker, then run the cell. Prefer explicitly selecting safe fields to hoping a generic serializer will redact every secret. The safe representation is a new value; the unsafe dictionary and string still exist in memory. This demonstrates prevention before retention, not deletion of an already-written log.

In [ ]:
request_metadata = {'caller':'partner','route':'/v1/diagnose','caller_key':MARKER}
unsafe_log = json.dumps(request_metadata)
safe_log = json.dumps({name:request_metadata[name] for name in ('caller','route')})
log_result = {'whole_request_contains_marker':marker_present(unsafe_log),'selected_fields_contain_marker':marker_present(safe_log)}
print(log_result)
assert log_result=={'whole_request_contains_marker':True,'selected_fields_contain_marker':False}

### Select task inputs for the model

The model needs the question and equipment context. It does not need proof that the caller authenticated. A prompt is another transmission boundary; carrying a key into it is exposure even when the final answer hides it.

In [ ]:
request = {'question':'Which filter checks are documented?','equipment_id':'EQ-FC-1002','caller_key':MARKER}
unsafe_prompt = json.dumps(request)
safe_prompt = json.dumps({name:request[name] for name in ('question','equipment_id')})
prompt_result = {'whole_request_contains_marker':marker_present(unsafe_prompt),'task_fields_contain_marker':marker_present(safe_prompt)}
print(prompt_result)
assert prompt_result=={'whole_request_contains_marker':True,'task_fields_contain_marker':False}

### Observe tracking and history as different states

The helper creates and deletes a disposable local Git repository. It commits a marker file, adds an ignore rule, then removes the file from tracking. It does not touch your own repository.

Predict: does adding `.gitignore` untrack an already committed file? Does removing a file from the current index erase old commits? Expect `True`, `False`, `True` for tracked-after-ignore, tracked-after-removal, and marker-still-in-history.

If this were a real credential, revoke/rotate it at its issuer, remove the exposed value from current files and relevant retained artifacts, and follow the organization's history-remediation process. Ignoring a filename alone cannot undo exposure.

In [ ]:
repository_result = repository_rehearsal()
print(repository_result)
assert repository_result['tracked_after_ignore']
assert not repository_result['tracked_after_remove']
assert repository_result['history_still_contains_marker']
evidence['secret_boundaries'] = {'logs':log_result,'prompt':prompt_result,'git':repository_result}
secret_note = ''  # Explain one exposure path and why its apparent fix is insufficient.

## 3. Worked example — bound accepted work fairly

Authentication limits **who** may call. A **rate limit** bounds admitted requests over time. It can reduce cost and burst abuse and keep one caller from using another's allocation; it does not by itself control model token cost or guarantee service availability.

| Bucket choice | What it groups | Tradeoff |
|---|---|---|
| Per key/caller | A recognized caller's requests | Separate callers stay independent; shared keys share an allowance |
| Per endpoint | Calls to one operation | Other versions may provide a bypass unless the intended scope is pooled |
| Per IP | A network address | People behind one gateway can share an allowance; addresses can change |

Here we use **one bucket per configured caller label across all protected POST versions**. The label, not the raw key string, identifies the bucket. Rotating its runtime key is a separate operation; this teaching service reads keys at startup, and restarting resets all in-memory counters. A fixed window starts with that caller's first admitted request and lasts `60` seconds. The worked allowance is `2`.

| Event | Status | Count/work consequence |
|---|---|---|
| Missing/wrong key | `401` | No allowance and no adapter work |
| Admitted valid input | Usually `200` | Uses one allowance; supported complete context enters the adapter |
| Admitted invalid input | `422` | Uses one allowance, but no adapter work |
| Exhausted allowance | `429` + `Retry-After` | No further count and no adapter work |
| Clarification or upstream failure after admission | `200` clarification / `502` | Still uses one allowance |

Counters are in-memory in one process. Restarting the server clears them; it is **not evidence that a timed window renewed**. Multiple workers would require shared state beyond this teaching implementation.

In [ ]:
limit_lab, _ = prepare_security_workspace(work/'limit-lab',checkpoint=auth,repo_root=REPO)
limit_settings = 'from module_b.security import LimitPolicy\n'+auth_settings.replace('POLICY = None','POLICY = LimitPolicy(allowance=2, window_seconds=60)')
write_source(limit_lab,'app/security_settings.py',limit_settings,expected=auth_settings)
print(limit_settings)

### Observe an actual burst

Predict the four statuses before running: dispatch, dispatch, dispatch, partner. The third dispatch request should be rejected while the independent partner still succeeds. `Retry-After` is the number of whole seconds to wait, rounded up.

In [ ]:
limit_http = []
with demo_runtime(limit_lab,env) as service:
    for caller in ('dispatch','dispatch','dispatch','partner'):
        response = httpx.post(service.base_url+'/v1/diagnose',json=accepted,headers={'X-API-Key':env[CALLER_ENV[caller]]})
        limit_http.append({'caller':caller,'status':response.status_code,'retry_after':response.headers.get('retry-after')})
table(limit_http)
assert [r['status'] for r in limit_http]==[200,200,429,200]

### Observe renewal without waiting or restarting

The separate probe has a controlled clock. `at` is elapsed seconds in one persistent worker, not a delay between cells. Dispatch begins at `0`; partner begins at `10`, so their windows end at different times.

Follow the sequence before running. Requests to the second inherited route share the same caller's bucket. At `60`, dispatch can renew while partner's first window remains active until `70`.

In [ ]:
second_path = next(p for p in paths if p!='/v1/diagnose')
worked_sequence = [
    {'caller':'dispatch','path':'/v1/diagnose','at':0},
    {'caller':'dispatch','path':second_path,'at':0},
    {'caller':'dispatch','path':'/v1/diagnose','at':0},
    {'caller':'partner','path':'/v1/diagnose','at':10},
    {'caller':'dispatch','path':'/v1/diagnose','at':60},
    {'caller':'partner','path':second_path,'at':60},
    {'caller':'partner','path':'/v1/diagnose','at':60},
    {'caller':'partner','path':'/v1/diagnose','at':70},
]
limit_observed = observe_security(limit_lab,caller_env=CALLER_ENV,sequence=worked_sequence)
table(limit_observed['rows'],['at','caller','path','status','adapter_calls','retry_after'])
assert [r['status'] for r in limit_observed['rows']]==[200,200,429,200,200,200,429,200]

### Check what “admitted” means

Now send an unauthenticated request, a recognized request with an empty body, and two valid recognized requests. Predict how validation changes work without refunding the admission. Expect `401`, `422`, `200`, `429` and adapter counts `0`, `0`, `1`, `0`.

In [ ]:
counting = observe_security(limit_lab,caller_env=CALLER_ENV,sequence=[
    {'caller':'missing','path':'/v1/diagnose'},
    {'caller':'dispatch','path':'/v1/diagnose','payload':{}},
    {'caller':'dispatch','path':'/v1/diagnose'},
    {'caller':'dispatch','path':'/v1/diagnose'},
])
table(counting['rows'],['caller','status','adapter_calls','retry_after'])
assert [r['status'] for r in counting['rows']]==[401,422,200,429]
assert [r['adapter_calls'] for r in counting['rows']]==[0,0,1,0]
evidence['limit_lab'] = {'http':limit_http,'timeline':limit_observed,'counting':counting}

## 4. Your turn — apply a per-caller policy

✂️ **Independent practice**

Dispatch and partner each need **3 admissions per 60-second first-admission window**, shared across all their route versions. Start from the authentication stage; preserve its routes, contracts, prompts and data. Change the policy in `app/security_settings.py`. Keep the existing caller names and environment-variable names.

Design a nondecreasing timeline that distinguishes authentication failure, admitted invalid input, exhaustion, independent callers, use across two routes and timed renewal after exhaustion. Write `practice_predictions` **before** running your chosen `practice_sequence`; retain the prediction if the observation surprises you and explain the difference. Include both missing and wrong callers. Include an exhausted caller whose admissions span two routes, then observe one caller renewing while the other is still in its original window and reaches or remains at its limit. Then choose a short actual HTTP burst that reaches `429` and demonstrates that the other caller can still succeed.

Supply complete source in `practice_edits` or edit the file directly at the printed path. Set `PRACTICE_READY=True` when ready to observe it. Empty input preserves your files and keeps the exercise pending.

<details><summary>Hint 1 — choose evidence for each claim</summary>A successful call cannot prove a limit. A rejected call cannot alone prove which boundary rejected it. Use status, caller, route, time and adapter count together.</details>
<details><summary>Hint 2 — distinguish shared and independent state</summary>Change routes while retaining a caller, then change callers while retaining the route. Predict which switch should retain the same bucket.</details>
<details><summary>Hint 3 — account for invalid input</summary>Authentication happens first, then admission, then input validation. A recognized malformed request can reduce the remaining allowance without entering the adapter.</details>

In [ ]:
practice, _ = prepare_security_workspace(work/'practice',checkpoint=auth,repo_root=REPO)
practice_base = baseline(practice,'limit-practice',paths)
print('Edit your practice service:',practice)

In [ ]:
practice_edits = {}  # {'app/security_settings.py': 'your complete source'}
apply_student_edits(practice,practice_edits)
PRACTICE_READY = False
practice_sequence = []  # Rows with caller, path, at; optional payload={} for invalid input.
practice_predictions = []
practice_http_callers = []
practice_http_path = '/v1/diagnose'
practice_note = ''  # Explain a rejection, caller fairness, timed renewal and evidence limits.

In [ ]:
practice_report = {'passed':False,'status':'pending'}
if PRACTICE_READY and practice_sequence and practice_http_callers:
    observed = observe_security(practice,caller_env=CALLER_ENV,sequence=practice_sequence)
    table(observed.get('rows',[]),['at','caller','path','status','adapter_calls','retry_after'])
    http_rows = []
    practice_env = runtime_keys(CALLER_ENV)
    with demo_runtime(practice,practice_env) as service:
        for caller in practice_http_callers:
            headers = {} if caller=='missing' else {'X-API-Key':practice_env[CALLER_ENV[caller]] if caller in CALLER_ENV else 'invalid-local-test'}
            response = httpx.post(service.base_url+practice_http_path,json=accepted,headers=headers)
            http_rows.append({'caller':caller,'path':practice_http_path,'status':response.status_code,'retry_after':response.headers.get('retry-after')})
    table(http_rows)
    practice_report = review_security(practice,caller_env=CALLER_ENV,allowance=3,sequence=practice_sequence,
        predictions=practice_predictions,observed=observed,http_rows=http_rows,inherited=practice_base)
    table([{'check':name,'passed':value} for name,value in practice_report['checks'].items()])
    table(practice_report['missing_configuration'])
print('Practice local checks:',practice_report['passed'])
evidence['practice'] = practice_report

## 5. Weekend service checkpoint

✂️ **Independent assessment**

The weekend operation has two callers: `operations` using `FIELDCARE_OPERATIONS_KEY`, and `weekend_partner` using `FIELDCARE_WEEKEND_PARTNER_KEY`. Each needs **4 admissions per 60-second first-admission window**, pooled across every existing protected POST route. Neither may consume the other's allowance.

**Task A — authenticate:** configure the new caller arrangement and keep secrets out of source, notebooks, logs, prompts and responses. Missing/wrong callers must stop before adapter work; missing required runtime configuration must refuse startup and accept calls again after restoration. In `configuration_template`, write one `NAME=<set-in-runtime>` line per required caller variable. Explain runtime loading and your secret inspection in `configuration_note`; no usable value belongs in the template.

**Task B — limit:** apply the new policy while preserving inherited route behavior and public GET health/docs. Explain admission accounting, route coverage, caller isolation and timed renewal. Keep the same single-process teaching scope.

**Task C — demonstrate:** choose your own predicted timeline and actual HTTP burst. Cover successful admission, missing/wrong caller, admitted invalid input, exhaustion with `Retry-After`, two route versions sharing one exhausted caller bucket, both callers and renewal after an exhausted window. Include an observation where one caller has renewed but the other remains in its original window and reaches or remains at its limit. Explain what actual HTTP proves and what the separate controlled-clock adapter observations add. Do not restart a server and call that renewal.

### Diagnose before adapting

In the worked two-request example, a dispatch request succeeds again after a fresh process starts. Does that prove the `60`-second window ended? Record your diagnosis in `assessment_note`, then build the new arrangement.

<details><summary>Solution — diagnose the worked example only</summary>

**Result you should get:** restart success does not prove timed renewal. **How:** the bucket lived in the old process. Use one controlled-clock worker with an exhausted bucket and a later time to observe actual window expiry.

</details>

The check below validates local observations and preservation; it does not award a grade or claim production security. Use the taught patterns and course reference material. Your instructor determines review/submission rules; no extra assessment policy is implied here.

In [ ]:
assessment = None
assessment_base = None
if practice_report.get('passed') and fresh(practice_report,practice):
    assessment, _ = prepare_security_workspace(work/'assessment',checkpoint=practice,repo_root=REPO)
    assessment_base = baseline(assessment,'secured-assessment',paths)
    print('Edit the weekend service:',assessment)
else:
    print('Pending: complete or rerun the current independent practice check first.')

In [ ]:
assessment_edits = {}  # Your own configuration; no solution is inserted.
if assessment is not None:
    apply_student_edits(assessment,assessment_edits)
ASSESSMENT_READY = False
ASSESSMENT_CALLERS = {'operations':'FIELDCARE_OPERATIONS_KEY','weekend_partner':'FIELDCARE_WEEKEND_PARTNER_KEY'}
assessment_sequence = []
assessment_predictions = []
assessment_http_callers = []
assessment_http_path = '/v1/diagnose'
assessment_note = ''
configuration_template = ''  # Variable names and <set-in-runtime> placeholders only.
configuration_note = ''      # Runtime loading, restoration and your source/evidence secret review.

In [ ]:
assessment_report = {'passed':False,'status':'pending'}
if ASSESSMENT_READY and assessment is not None and assessment_sequence and assessment_http_callers:
    observed = observe_security(assessment,caller_env=ASSESSMENT_CALLERS,sequence=assessment_sequence)
    table(observed.get('rows',[]),['at','caller','path','status','adapter_calls','retry_after'])
    http_rows = []
    assessment_env = runtime_keys(ASSESSMENT_CALLERS)
    with demo_runtime(assessment,assessment_env) as service:
        for caller in assessment_http_callers:
            headers = {} if caller=='missing' else {'X-API-Key':assessment_env[ASSESSMENT_CALLERS[caller]] if caller in ASSESSMENT_CALLERS else 'invalid-local-test'}
            response = httpx.post(service.base_url+assessment_http_path,json=accepted,headers=headers)
            http_rows.append({'caller':caller,'path':assessment_http_path,'status':response.status_code,'retry_after':response.headers.get('retry-after')})
    table(http_rows)
    assessment_report = review_security(assessment,caller_env=ASSESSMENT_CALLERS,allowance=4,sequence=assessment_sequence,
        predictions=assessment_predictions,observed=observed,http_rows=http_rows,inherited=assessment_base)
    table([{'check':name,'passed':value} for name,value in assessment_report['checks'].items()])
    table(assessment_report['missing_configuration'])
print('Assessment local checks:',assessment_report['passed'],'— instructor review remains pending.')
evidence['assessment'] = assessment_report

## 6. Save one final checkpoint

Your final handoff contains the most advanced stage available, the actual source, predictions, observations and your explanations. Earlier stages remain in this runtime to support learning; you manage only **this notebook and one final source ZIP**. Save the notebook to keep your cell edits and notes.

Before sharing, check that no allowed source/JSON file contains a credential. Environment files are excluded, but an exporter cannot undo a pasted secret. Download the printed ZIP before a Colab runtime ends. A pending export still preserves useful incomplete work; its readiness record stays honest.

If you change source or a chosen experiment after checking it, rerun the check. The summary below ties current readiness to both the source and the selected experiment. All results here are local demo observations; no hosted deployment or live-provider behavior has been verified.

In [ ]:
from module_b.live_sessions import read_records
assessment=globals().get('assessment')
practice=globals().get('practice',auth)
ASSESSMENT_CALLERS=globals().get('ASSESSMENT_CALLERS',{'operations':'FIELDCARE_OPERATIONS_KEY','weekend_partner':'FIELDCARE_WEEKEND_PARTNER_KEY'})
assessment_report=globals().get('assessment_report',{'passed':False,'status':'pending'})
assessment_sequence=globals().get('assessment_sequence',[])
assessment_predictions=globals().get('assessment_predictions',[])
assessment_http_callers=globals().get('assessment_http_callers',[])
assessment_http_path=globals().get('assessment_http_path','/v1/diagnose')
secret_note=globals().get('secret_note','')
practice_note=globals().get('practice_note','')
assessment_note=globals().get('assessment_note','')
configuration_template=globals().get('configuration_template','')
configuration_note=globals().get('configuration_note','')
# Preserve a resumed assessment even if an earlier task still needs its check rerun.
final_project = assessment if assessment is not None else work/'assessment' if (work/'assessment').exists() else practice
final_stage = 'assessment' if final_project.name=='assessment' else 'auth' if final_project==auth else 'practice'
selection = {'caller_env':ASSESSMENT_CALLERS,'sequence':assessment_sequence,'predictions':assessment_predictions,
             'http_callers':assessment_http_callers,'http_path':assessment_http_path}
template_ready = configuration_template_ok(configuration_template,ASSESSMENT_CALLERS)
readiness = {'structural_ready':bool(assessment_report.get('passed') and fresh(assessment_report,final_project,selection)),
             'written_evidence_present':bool(secret_note.strip() and practice_note.strip() and assessment_note.strip() and configuration_note.strip()),
             'configuration_template_ready':template_ready,
             'mode':'local_demo','stage':final_stage,'instructor_review':'pending'}
live_records={**read_records(final_project),**read_records(auth)}
evidence['live_sessions']=live_records
if live_records:
    (final_project/'fixtures/live-session-records.json').write_text(json.dumps(live_records,indent=2)+'\n')
SPRINT2_EXPORT = export_progress(final_project,REPO/'artifacts',sprint=2,evidence=evidence,
    notes={'secret_boundaries':secret_note,'practice':practice_note,'assessment':assessment_note,
           'configuration_template':configuration_template if template_ready else '',
           'configuration_note':configuration_note},readiness=readiness)
print('Resume stage:',final_stage)
print('Readiness:',readiness)
if configuration_template and not template_ready:
    print('Template not exported: use exactly the required variable names and <set-in-runtime> placeholders.')

## Watch out for

| Observation | Check and next action |
|---|---|
| A protected version still returns `200` without a key | Check inventory and attach the guard after all registrations. |
| All callers start failing at the same count | Check whether the bucket is keyed by recognized caller. |
| Switching versions restores allowance | Check whether state is incorrectly keyed by endpoint. |
| A malformed authenticated request reduces allowance | Expected: admission precedes input validation. |
| A restart restores allowance | Expected for process memory; use timed observations to test renewal. |
| A key is missing from the runtime | Supply it privately; do not remove the fail-closed check. |
| An experiment is pending or fails | Read the specific checks, revise your source/predictions and rerun. |
| You resume into the wrong stage | Preserve the ZIP; use a new deliberate run name and the printed stage. |

## Quick reference

| Tool/pattern | Purpose |
|---|---|
| `CALLER_ENV` | Map recognized labels to runtime variable names, never values |
| `SecurityMiddleware` + `protected_post_paths` | Apply the taught guard to all registered POST operations |
| `LimitPolicy` | Express allowance and window duration |
| `runtime_keys` | Generate temporary local-only values without displaying them |
| `httpx.post(..., headers=...)` | Make the authentication boundary visible in an actual request |
| `observe_security` | Count adapter entries and control time in a separate no-network probe |
| `review_security` | Check your chosen evidence, route coverage and source preservation |
| `export_progress` | Save one source/evidence checkpoint |

## Summary

You controlled who can reach FieldCare, kept credentials out of downstream task data, and bounded admissions independently per caller. Your preserved service and saved evidence are ready to carry into the next sprint, with the limits of local single-process checks stated explicitly.